In [1]:
#!/usr/bin/env python3
"""
export_a1_lut.py — Precompute the A1 coordinate warp grid as a lookup table.

Since A1 is a pure coordinate→coordinate MLP (input-independent), its output
grid is identical for every image at a given resolution. This script runs the
MLP once and saves the resulting grid so C++ can skip the MLP entirely.

Usage:
  python3 tools/export_a1_lut.py --weights spatons_weights.bin --height 256 --width 256
  python3 tools/export_a1_lut.py --weights spatons_weights.bin -o a1_lut.bin

LUT binary format:
  [uint32] magic   = 0x41314C54 ("A1LT")
  [uint32] version = 1
  [uint32] H
  [uint32] W
  [float32 × H × W × 2]  grid values in [-1, 1], row-major, (x, y) per pixel
"""

'\nexport_a1_lut.py — Precompute the A1 coordinate warp grid as a lookup table.\n\nSince A1 is a pure coordinate→coordinate MLP (input-independent), its output\ngrid is identical for every image at a given resolution. This script runs the\nMLP once and saves the resulting grid so C++ can skip the MLP entirely.\n\nUsage:\n  python3 tools/export_a1_lut.py --weights spatons_weights.bin --height 256 --width 256\n  python3 tools/export_a1_lut.py --weights spatons_weights.bin -o a1_lut.bin\n\nLUT binary format:\n  [uint32] magic   = 0x41314C54 ("A1LT")\n  [uint32] version = 1\n  [uint32] H\n  [uint32] W\n  [float32 × H × W × 2]  grid values in [-1, 1], row-major, (x, y) per pixel\n'

In [2]:

import argparse
import struct
import os
import numpy as np


In [3]:
WEIGHTS_MAGIC = 0x5350544E  # "SPTN"
LUT_MAGIC     = 0x41314C54  # "A1LT"
LUT_VERSION   = 1

In [4]:
A1_HIDDEN = 64
A1_IN     = 2
A1_OUT    = 2

In [8]:
WEIGHTS_NAME = "#86-NN2_best.pth"

In [9]:
import torch
ckpt = torch.load(WEIGHTS_NAME, map_location='cpu')
print(type(ckpt))

<class 'dict'>


In [ ]:
print(type(ckpt))
if isinstance(ckpt, dict):
    for k, v in ckpt.items():
        try:    print(k, tuple(v.shape))
        except: print(k, type(v))   # non-tensor entries (epoch, optimizer, etc.)

In [ ]:
sd = ckpt['model_state_dict']
for k, v in sd.items():
    try:    print(k, tuple(v.shape))
    except: print(k, type(v))

In [7]:
ckpt['args']

NameError: name 'ckpt' is not defined

In [ ]:
ckpt['projector_config']

{'color_pipeline': [{'type': 'elevated_black_level', 'strength': 0.2},
  {'type': 'brightness_both_fixed', 'strength': 0.2},
  {'type': 'contrast_fixed', 'strength': -0.2},
  {'type': 'blur', 'strength': 0.2},
  {'type': 'lsnu', 'strength': 0.2}],
 'geometrical_pipeline': [{'type': 'keystone', 'strength': 0.3}]}

In [10]:
def load_a1_weights(WEIGHTS_NAME):
    ckpt = torch.load(WEIGHTS_NAME, map_location='cpu')
    sd = ckpt['model_state_dict']
    W1 = sd['models.0.model.layer1.weight'].numpy()
    b1 = sd['models.0.model.layer1.bias'].numpy()
    W2 = sd['models.0.model.layer2.weight'].numpy()
    b2 = sd['models.0.model.layer2.bias'].numpy()
    W3 = sd['models.0.model.layer3.weight'].numpy()
    b3 = sd['models.0.model.layer3.bias'].numpy()
    return W1, b1, W2, b2, W3, b3

In [11]:
def build_coord_grid(H, W):
    """Normalised (x, y) coordinate grid matching ApproachA1::forward."""
    ys = np.linspace(0, 1, H, dtype=np.float32) if H > 1 else np.array([0.5], dtype=np.float32)
    xs = np.linspace(0, 1, W, dtype=np.float32) if W > 1 else np.array([0.5], dtype=np.float32)
    yy, xx = np.meshgrid(ys, xs, indexing='ij')
    coords = np.stack([xx.ravel(), yy.ravel()], axis=1)  # [H*W, 2]
    return coords

In [12]:
def forward_a1_mlp(coords, W1, b1, W2, b2, W3, b3):
    """Replicate ParametricModel forward with identity residual."""
    h1 = np.maximum(0, coords @ W1.T + b1)
    h2 = np.maximum(0, h1 @ W2.T + b2)
    out = h2 @ W3.T + b3 + coords  # identity residual (in_dim == out_dim)
    return out


In [13]:
def main():
    p = argparse.ArgumentParser(description='Export A1 coordinate LUT')
    p.add_argument('--weights', required=True, help='Path to spatons_weights.bin')
    p.add_argument('--height', type=int, default=256)
    p.add_argument('--width',  type=int, default=256)
    p.add_argument('-o', '--output', default=None,
                   help='Output path (default: a1_lut_HxW.bin)')
    args = p.parse_args()

    H, W = 256, 256
    out_path = "lut.hex"

    W1, b1, W2, b2, W3, b3 = load_a1_weights("#86-NN2_best.pth")
    coords = build_coord_grid(H, W)
    transformed = forward_a1_mlp(coords, W1, b1, W2, b2, W3, b3)
    transformed = np.clip(transformed, 0.0, 1.0)

    # --- UNITS DECISION (see note) ---
    # Option A: pixel coordinates [0, W-1]  — for coord_split that floors pixel coords
    grid = (transformed * (W - 1)).astype(np.float32).reshape(H, W, 2)

    # Option B: [-1,1] grid_sample convention (the script's original)
    # grid = (2.0 * transformed - 1.0).astype(np.float32).reshape(H, W, 2)

    print(f"min={grid.min():.4f} max={grid.max():.4f} mean={grid.mean():.4f}")

    # write YOUR RTL hex packing: 64-bit {y,x} per line, row-major
    import struct
    with open(out_path, "w") as f:
        for y in range(H):
            for x in range(W):
                xf, yf = float(grid[y, x, 0]), float(grid[y, x, 1])
                xbits = struct.unpack('<I', struct.pack('<f', xf))[0]
                ybits = struct.unpack('<I', struct.pack('<f', yf))[0]
                f.write(f"{ybits:08x}{xbits:08x}\n")

    print(f"wrote {H*W} entries to {out_path}")

In [14]:
import sys

if __name__ == '__main__':
    # Save the original sys.argv
    original_argv = sys.argv

    try:
        # Simulate command-line arguments for argparse
        # IMPORTANT: Replace 'path/to/your/spatons_weights.bin' with the actual path
        # to your weights file. You can also specify --height and --width if needed.
        sys.argv = ['colab_kernel_launcher.py', '--weights', 'path/to/your/spatons_weights.bin', '--height', '256', '--width', '256']
        main()
    finally:
        # Restore sys.argv to its original state to avoid interfering with other notebook operations
        sys.argv = original_argv

min=0.0000 max=255.0000 mean=119.0324
wrote 65536 entries to lut.hex


In [15]:
import numpy as np


In [16]:
import numpy as np, struct

H = W = 256
grid = np.zeros((H, W, 2), dtype=np.float32)
with open("lut.hex") as f:
    for y in range(H):
        for x in range(W):
            line = f.readline().strip()         # "{ybits}{xbits}", 16 hex chars
            ybits = int(line[0:8], 16)           # high 8 hex = y
            xbits = int(line[8:16], 16)          # low 8 hex = x
            grid[y, x, 0] = struct.unpack('<f', struct.pack('<I', xbits))[0]  # x
            grid[y, x, 1] = struct.unpack('<f', struct.pack('<I', ybits))[0]  # y


In [ ]:
print("loaded grid:", grid.shape, "min", grid.min(), "max", grid.max())


loaded grid: (256, 256, 2) min 0.0 max 255.0


In [17]:
# reload what you generated, or use `grid` if still in memory
ys, xs = np.meshgrid(np.arange(256), np.arange(256), indexing='ij')
identity = np.stack([xs, ys], axis=-1).astype(np.float32)
disp = np.abs(grid - identity)

In [18]:
print("center (128,128):", grid[128,128])      # expect ~[127,127]
print("corner (0,0):    ", grid[0,0])
print("corner (255,255):", grid[255,255])
print("mean displacement:", disp.mean(), " max:", disp.max())

center (128,128): [128.68127  96.40782]
corner (0,0):     [18.582932  0.      ]
corner (255,255): [255. 255.]
mean displacement: 16.782764  max: 60.708927


In [21]:
for (x, y) in [(0,0), (128,128), (255,255), (5,3), (200,100)]:
    gx, gy = grid[y, x]      # note: grid[y,x] — row=y, col=x
    print(f"(x={x}, y={y}) -> x={gx:.9f}  y={gy: 9f}")

(x=0, y=0) -> x=18.582931519  y= 0.000000
(x=128, y=128) -> x=128.681274414  y= 96.407822
(x=255, y=255) -> x=255.000000000  y= 255.000000
(x=5, y=3) -> x=20.378252029  y= 0.000000
(x=200, y=100) -> x=207.071166992  y= 61.721947
